# Multilingual Brand-Safety Comparison

Load the consolidated LLM ratings and inspect where brand-safety or suitability opinions diverge across language versions of the same article.

In [ ]:
from pathlib import Path

import pandas as pd
# data file is in the repository root
DATA_PATH = Path("euronews_llm_ratings_consolidated_gemini_1000.csv")
df = pd.read_csv(DATA_PATH)
df


In [ ]:
# perform an analysis of variance for between run and between language differences in floor/level ratings (stable categories, titles with >=2 languages)
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

stable_df = df.copy()

title_lang_counts = stable_df.groupby('rss_title')['language'].nunique()
multi_lang_titles = title_lang_counts[title_lang_counts >= 2].index
anova_df = stable_df[stable_df['rss_title'].isin(multi_lang_titles)].copy()

# map ratings to numeric scores for averaging and ANOVA
floor_map = {'safe': 0, 'unsafe': 1}
level_map = {'0': 0, 'low': 1, 'medium': 2, 'high': 3}
anova_df['floor_score'] = anova_df['floor'].astype(str).str.lower().map(floor_map)
anova_df['level_score'] = anova_df['level'].astype(str).str.lower().map(level_map)

anova_df = anova_df.dropna(subset=['language', 'run_index', 'category', 'rss_title']).copy()
anova_df['run_label'] = 'run' + anova_df['run_index'].astype(int).astype(str)
floor_anova_df = anova_df.dropna(subset=['floor_score']).copy()
level_anova_df = anova_df.dropna(subset=['level_score']).copy()

# mean floor and level scores by run and language (rows=runs, cols=languages)
floor_table = floor_anova_df.pivot_table(index='run_label', columns='language', values='floor_score', aggfunc='mean')
level_table = level_anova_df.pivot_table(index='run_label', columns='language', values='level_score', aggfunc='mean')

display(floor_table.style.format('{:.3f}').set_caption('Mean floor score (0=safe, 1=unsafe) by run and language'))
display(level_table.style.format('{:.3f}').set_caption('Mean level score (0-3) by run and language'))

# two-way ANOVA (run, language, and their interaction) on the numeric scores
floor_model = ols('floor_score ~ C(run_label) * C(language)', data=floor_anova_df).fit()
level_model = ols('level_score ~ C(run_label) * C(language)', data=level_anova_df).fit()

floor_anova = anova_lm(floor_model, typ=2)
level_anova = anova_lm(level_model, typ=2)

display(floor_anova.style.format('{:.3f}').set_caption('Two-way ANOVA for floor scores (0=safe, 1=unsafe)'))
display(level_anova.style.format('{:.3f}').set_caption('Two-way ANOVA for level scores (0-3)'))

# full fixed-effects ANOVA: run, language, category, title (rss_title), and language-by-category interaction
# This tests whether language differences remain after controls, and whether they vary by category.
full_formula_floor = 'floor_score ~ C(run_label) + C(language) + C(category) + C(language):C(category) + C(rss_title)'
full_formula_level = 'level_score ~ C(run_label) + C(language) + C(category) + C(language):C(category) + C(rss_title)'

full_floor_model = ols(full_formula_floor, data=floor_anova_df).fit()
full_level_model = ols(full_formula_level, data=level_anova_df).fit()

full_floor_anova = anova_lm(full_floor_model, typ=2)
full_level_anova = anova_lm(full_level_model, typ=2)

display(full_floor_anova.style.format('{:.3f}').set_caption('Full fixed-effects ANOVA for floor scores: run + language + category + language:category + title'))
display(full_level_anova.style.format('{:.3f}').set_caption('Full fixed-effects ANOVA for level scores: run + language + category + language:category + title'))

In [ ]:
# language coverage summary (run-collapsed) with title count + avg body length
import gzip
import json

num_unique_titles = df["rss_title"].nunique()
print(f"Number of unique titles in consolidated ratings CSV: {num_unique_titles}")

OUT_JSONL_PATH = Path('euronews_brand_safety_publication_dataset.jsonl.gz')
SOURCE_ARTICLES_PATH = Path('euronews_multilingual_consolidated_dedup_apr2_10773.json')

# 1) run-collapsed language versions from output JSONL
records = []
open_jsonl = gzip.open if OUT_JSONL_PATH.suffix == '.gz' else open
with open_jsonl(OUT_JSONL_PATH, 'rt', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        records.append({
            'article_index': rec.get('article_index'),
            'language': rec.get('language'),
            'rss_title': rec.get('rss_title'),
        })

pairs_df = (
    pd.DataFrame(records)
      .dropna(subset=['article_index', 'language'])
      .drop_duplicates(subset=['article_index', 'language'])
      .copy()
)

# fill missing titles from consolidated CSV if needed
if pairs_df['rss_title'].isna().any():
    title_map = (
        df[['article_index', 'language', 'rss_title']]
        .dropna(subset=['article_index', 'language', 'rss_title'])
        .drop_duplicates(subset=['article_index', 'language'])
    )
    pairs_df = pairs_df.merge(
        title_map,
        on=['article_index', 'language'],
        how='left',
        suffixes=('', '_from_csv')
    )
    pairs_df['rss_title'] = pairs_df['rss_title'].fillna(pairs_df['rss_title_from_csv'])
    pairs_df = pairs_df.drop(columns=['rss_title_from_csv'])

# 2) body word counts from source multilingual article file
if SOURCE_ARTICLES_PATH.exists():
    source_obj = json.loads(SOURCE_ARTICLES_PATH.read_text(encoding='utf-8'))
else:
    source_obj = {'results': []}
body_rows = []
for article in source_obj.get('results', []):
    article_index = article.get('article_index')
    article_title = article.get('rss_title')
    for version in article.get('versions', []):
        language = version.get('language')
        if article_index is None or language is None:
            continue
        word_count = version.get('word_count')
        if word_count is None:
            body = version.get('body') or ''
            word_count = len(str(body).split())
        body_rows.append({
            'article_index': article_index,
            'language': language,
            'rss_title': article_title,
            'body_word_count': word_count,
        })

body_cols = ['article_index', 'language', 'rss_title', 'body_word_count']
body_df = pd.DataFrame(body_rows, columns=body_cols).dropna(subset=['article_index', 'language'])
body_by_index = body_df.drop_duplicates(subset=['article_index', 'language'], keep='first')
body_by_title = body_df.dropna(subset=['rss_title']).drop_duplicates(subset=['rss_title', 'language'], keep='first')

# primary join: (article_index, language)
pairs_with_body = pairs_df.merge(
    body_by_index[['article_index', 'language', 'body_word_count']],
    on=['article_index', 'language'],
    how='left'
)

# fallback join: (rss_title, language) for index mismatches
missing_mask = pairs_with_body['body_word_count'].isna()
if missing_mask.any():
    fallback = pairs_with_body.loc[missing_mask, ['rss_title', 'language']].merge(
        body_by_title[['rss_title', 'language', 'body_word_count']],
        on=['rss_title', 'language'],
        how='left'
    )
    pairs_with_body.loc[missing_mask, 'body_word_count'] = fallback['body_word_count'].to_numpy()

# 3) requested per-language table
language_counts = pairs_df['language'].value_counts().rename('language_version_count')
distinct_title_counts = pairs_df.groupby('language')['rss_title'].nunique().rename('distinct_title_count')
avg_body_words = pairs_with_body.groupby('language')['body_word_count'].mean().rename('avg_news_body_words')

language_summary = (
    pd.concat([language_counts, distinct_title_counts, avg_body_words], axis=1)
      .reset_index()
      .rename(columns={'index': 'language'})
      .sort_values('language_version_count', ascending=False)
)
language_summary['avg_news_body_words'] = language_summary['avg_news_body_words'].round(1)

# keep legacy variable names for downstream cells
language_counts_per_15 = language_counts
total_count_per_15 = int(language_counts_per_15.sum())

print(f"Total language versions (run-collapsed): {total_count_per_15}")
print(f"Rows missing body_word_count after fallback merge: {pairs_with_body['body_word_count'].isna().sum()}")

display(language_summary)
# save to CSV for downstream use
language_summary.to_csv('../data/language_summary_apr2.csv', index=False)



In [ ]:
# Language discrepancy percentages by category (avg suitability level across runs)
score_level = {'0': 0, 'low': 1, 'medium': 2, 'high': 3}

avg_level = (
    df.loc[:, ['rss_title', 'language', 'category', 'level', 'run_index']]
      .dropna(subset=['rss_title', 'language', 'category', 'level'])
      .assign(level_score=lambda x: x['level'].astype(str).str.lower().map(score_level))
      .dropna(subset=['level_score'])
      .groupby(['rss_title', 'category', 'language'], as_index=False)['level_score']
      .mean()
      .rename(columns={'level_score': 'avg_level'})
)

# A language entry is discrepant if, for the same title+category,
# at least one other language has a different avg_level.
group_stats = (
    avg_level.groupby(['rss_title', 'category'])['avg_level']
    .agg(n_lang='size', group_min='min', group_max='max')
    .reset_index()
)

comp = avg_level.merge(group_stats, on=['rss_title', 'category'], how='left')
comp = comp[comp['n_lang'] >= 2].copy()
comp['is_discrepant'] = (comp['group_max'] - comp['group_min']) > 1e-12

discrepancy_pct = (
    comp.groupby(['category', 'language'], as_index=False)['is_discrepant']
      .mean()
      .assign(discrepancy_pct=lambda x: x['is_discrepant'] * 100)
      .drop(columns='is_discrepant')
)

discrepancy_table = (
    discrepancy_pct
    .pivot(index='category', columns='language', values='discrepancy_pct')
    .sort_index()
)

discrepancy_table_display = discrepancy_table.apply(lambda col: col.map(lambda v: f"{v:.1f}%" if pd.notna(v) else ""))

display(discrepancy_table_display)

out_path = Path('language_discrepancy_percentages_by_category.csv')
discrepancy_table_display.to_csv(out_path)
print(f"Saved discrepancy table to: {out_path.resolve()}")





In [ ]:
#  do plot heatmap of discrepancy percentages by category and language
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(12, 6), dpi=600)
sns.heatmap(discrepancy_table, annot=True, fmt=".1f", cmap
            =sns.color_palette("Reds", as_cmap=True), cbar_kws={'label': 'Discrepancy %'})
plt.title('Language Discrepancy Percentages by Category')
plt.xlabel('Language')
plt.ylabel('Category')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Simple table replacing Figure 1: article-category disagreement rates by category
raw_col = "% of Articles w/ 1+ Language Disagreements"
cond_col = "% of Articles w language disagreement given at least 1 non-0 rating"

article_cat = (
    avg_level.groupby(['rss_title', 'category'], as_index=False)
    .agg(
        n_languages=('language', 'nunique'),
        min_avg_level=('avg_level', 'min'),
        max_avg_level=('avg_level', 'max'),
        has_nonzero=('avg_level', lambda s: (s > 0).any()),
    )
)

article_cat = article_cat[article_cat['n_languages'] >= 2].copy()
article_cat['has_lang_disagreement'] = (
    article_cat['max_avg_level'] - article_cat['min_avg_level']
) > 1e-12

if article_cat.empty:
    simple_table = pd.DataFrame(columns=[raw_col, cond_col])
    simple_table.loc['TOTAL', [raw_col, cond_col]] = [pd.NA, pd.NA]
else:
    raw_pct = article_cat.groupby('category')['has_lang_disagreement'].mean().mul(100)
    cond_den = article_cat.groupby('category')['has_nonzero'].sum()
    cond_num = (
        article_cat[article_cat['has_nonzero']]
        .groupby('category')['has_lang_disagreement']
        .sum()
    )
    cond_pct = cond_num.div(cond_den.replace(0, pd.NA)).mul(100)

    simple_table = pd.DataFrame({
        raw_col: raw_pct,
        cond_col: cond_pct.reindex(raw_pct.index),
    }).sort_index()

    simple_table.loc['TOTAL', raw_col] = article_cat['has_lang_disagreement'].mean() * 100
    total_nonzero = article_cat['has_nonzero'].sum()
    simple_table.loc['TOTAL', cond_col] = (
        article_cat.loc[article_cat['has_nonzero'], 'has_lang_disagreement'].mean() * 100
        if total_nonzero > 0 else pd.NA
    )

simple_table_display = simple_table.copy()
for col in simple_table_display.columns:
    simple_table_display[col] = simple_table_display[col].map(
        lambda v: f"{v:.1f}%" if pd.notna(v) else "N/A"
    )

simple_table_display.index.name = 'Category'
display(simple_table_display.reset_index())

out_path = Path('language_disagreement_summary_table.csv')
simple_table_display.to_csv(out_path, index=True)
print(f"Saved summary table to: {out_path.resolve()}")


In [ ]:
# v2 with significance tests for avg_run_lang_matrix
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp, wilcoxon
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Avg-run multilingual disparity analyses
# Uses per-category mean across runs instead of stability filter
# + statistical significance tests for avg_run_lang_matrix
# ============================================================

score_floor = {'safe': 0, 'unsafe': 1}
score_level = {'0': 0, 'low': 1, 'medium': 2, 'high': 3}

# ------------------------------------------------------------
# 1) Recompute from raw data so we include all runs (no stability filter)
# ------------------------------------------------------------
avg_run = (
    pd.read_csv(DATA_PATH)
      .loc[:, ['rss_title', 'language', 'category', 'floor', 'level', 'run_index']]
      .dropna(subset=['rss_title', 'language', 'category', 'floor', 'level'])
)

# keep only language versions ending not with "_mt" (machine-translated)
avg_run = avg_run[~avg_run['language'].str.endswith('_mt')]

avg_run = (
    avg_run
    .assign(
        floor_score=avg_run['floor'].astype(str).str.lower().map(score_floor),
        level_score=avg_run['level'].astype(str).str.lower().map(score_level),
    )
    .dropna(subset=['floor_score', 'level_score'])
    .groupby(['rss_title', 'language', 'category'], as_index=False)
    .agg(
        avg_floor=('floor_score', 'mean'),
        avg_level=('level_score', 'mean'),
        runs_observed=('run_index', 'nunique'),
    )
)

# Keep titles with 2+ languages overall
multi_titles = avg_run.groupby('rss_title')['language'].nunique()
avg_run_multi = avg_run[avg_run['rss_title'].isin(multi_titles[multi_titles >= 2].index)].copy()

display(avg_run.head())
display(avg_run_multi.head())

# ------------------------------------------------------------
# 2) Pairwise avg_level deltas (unordered language pairs)
#    This reproduces your earlier overall pairwise summary
# ------------------------------------------------------------
avg_run_pairs = []

for title, g in avg_run_multi.groupby('rss_title'):
    langs = g['language'].unique()
    for la, lb in combinations(langs, 2):
        a_scores = g.loc[g['language'] == la, 'avg_level'].values
        b_scores = g.loc[g['language'] == lb, 'avg_level'].values
        for a in a_scores:
            for b in b_scores:
                avg_run_pairs.append({
                    'lang_i': la,
                    'lang_j': lb,
                    'd': a - b
                })

if avg_run_pairs:
    avg_run_pair_stats = (
        pd.DataFrame(avg_run_pairs)
        .groupby(['lang_i', 'lang_j'])['d']
        .agg(
            n='count',
            mean='mean',
            p_gt0=lambda s: (s > 0).mean().round(2),
            p_lt0=lambda s: (s < 0).mean().round(2)
        )
        .reset_index()
        .sort_values('mean', ascending=False)
        .round({'mean': 3})
    )
else:
    avg_run_pair_stats = pd.DataFrame(
        columns=['lang_i', 'lang_j', 'n', 'mean', 'p_gt0', 'p_lt0']
    )

display(avg_run_pair_stats)

# ------------------------------------------------------------
# 3) Overall language ranking on avg_level (positive = harsher)
# ------------------------------------------------------------
if avg_run_pair_stats.empty:
    avg_run_lang_rank = pd.DataFrame(columns=['language', 'avg_level_shift_vs_others'])
else:
    score_sum = {}
    weight_sum = {}

    for _, r in avg_run_pair_stats.iterrows():
        w = r['n']
        li, lj = r['lang_i'], r['lang_j']
        delta = r['mean']

        score_sum[li] = score_sum.get(li, 0) + delta * w
        score_sum[lj] = score_sum.get(lj, 0) - delta * w
        weight_sum[li] = weight_sum.get(li, 0) + w
        weight_sum[lj] = weight_sum.get(lj, 0) + w

    rank_rows = []
    for lang, s in score_sum.items():
        w = weight_sum.get(lang, 0)
        avg_shift = s / w if w else 0
        rank_rows.append({
            'language': lang,
            'avg_level_shift_vs_others': round(avg_shift, 4)
        })

    avg_run_lang_rank = (
        pd.DataFrame(rank_rows)
        .sort_values('avg_level_shift_vs_others', ascending=False)
        .reset_index(drop=True)
    )

display(avg_run_lang_rank)

# ------------------------------------------------------------
# 4) Title-level matched shifts within category
#    This is the aligned estimator for avg_run_lang_matrix:
#    shift_vs_others = avg_level(lang) - mean(avg_level(other langs))
#    computed within the same title and same category.
# ------------------------------------------------------------
title_level_rows = []

for cat, cat_df in avg_run_multi.groupby('category'):
    for title, g in cat_df.groupby('rss_title'):
        # Need at least 2 languages in this title/category
        if g['language'].nunique() < 2:
            continue

        lang_to_score = g.groupby('language', as_index=True)['avg_level'].mean().to_dict()
        langs = list(lang_to_score.keys())

        for lang in langs:
            others = [lang_to_score[l] for l in langs if l != lang]
            if len(others) == 0:
                continue

            shift = lang_to_score[lang] - np.mean(others)

            title_level_rows.append({
                'category': cat,
                'rss_title': title,
                'language': lang,
                'avg_level': lang_to_score[lang],
                'shift_vs_others': shift,
                'n_langs_in_title': len(langs),
            })

title_level_shift_df = pd.DataFrame(title_level_rows)

display(title_level_shift_df.head())

# ------------------------------------------------------------
# 5) Per-category language ranking from title-level shifts
#    This is the main, test-aligned quantity
# ------------------------------------------------------------
if title_level_shift_df.empty:
    avg_run_cat_lang_rank = pd.DataFrame(
        columns=['category', 'language', 'avg_level_shift_vs_others', 'n_titles']
    )
    avg_run_lang_matrix = pd.DataFrame()
else:
    avg_run_cat_lang_rank = (
        title_level_shift_df
        .groupby(['category', 'language'], as_index=False)
        .agg(
            avg_level_shift_vs_others=('shift_vs_others', 'mean'),
            n_titles=('rss_title', 'nunique')
        )
    )

    avg_run_cat_lang_rank['avg_level_shift_vs_others'] = (
        avg_run_cat_lang_rank['avg_level_shift_vs_others'].round(3)
    )

    avg_run_lang_matrix = (
        avg_run_cat_lang_rank
        .pivot(index='category', columns='language', values='avg_level_shift_vs_others')
        .sort_index()
    )

display(avg_run_cat_lang_rank)
display(avg_run_lang_matrix)

# ------------------------------------------------------------
# 6) Statistical significance tests for each cell in avg_run_lang_matrix
#    H0: mean(shift_vs_others) = 0
# ------------------------------------------------------------
sig_rows = []

for (cat, lang), g in title_level_shift_df.groupby(['category', 'language']):
    x = g['shift_vs_others'].dropna().values
    n = len(x)

    mean_shift = np.mean(x) if n > 0 else np.nan
    median_shift = np.median(x) if n > 0 else np.nan
    sd_shift = np.std(x, ddof=1) if n >= 2 else np.nan

    # one-sample t-test against 0
    if n >= 2 and np.isfinite(sd_shift) and sd_shift > 0:
        t_stat, t_p = ttest_1samp(x, popmean=0, alternative='two-sided')
    else:
        t_stat, t_p = np.nan, np.nan

    # wilcoxon signed-rank against 0
    try:
        if n >= 1 and np.any(x != 0):
            w_stat, w_p = wilcoxon(x, alternative='two-sided', zero_method='wilcox')
        else:
            w_stat, w_p = np.nan, np.nan
    except Exception:
        w_stat, w_p = np.nan, np.nan

    sig_rows.append({
        'category': cat,
        'language': lang,
        'n_titles': n,
        'mean_shift': mean_shift,
        'median_shift': median_shift,
        'sd_shift': sd_shift,
        't_stat': t_stat,
        't_p': t_p,
        'wilcoxon_stat': w_stat,
        'wilcoxon_p': w_p,
    })

avg_run_lang_sig = pd.DataFrame(sig_rows)

# ------------------------------------------------------------
# 7) Multiple-testing correction (BH/FDR)
# ------------------------------------------------------------
for p_col in ['t_p', 'wilcoxon_p']:
    avg_run_lang_sig[p_col + '_fdr'] = np.nan
    avg_run_lang_sig[p_col + '_sig'] = False

    mask = avg_run_lang_sig[p_col].notna()
    if mask.sum() > 0:
        reject, p_adj, _, _ = multipletests(avg_run_lang_sig.loc[mask, p_col], method='fdr_bh')
        avg_run_lang_sig.loc[mask, p_col + '_fdr'] = p_adj
        avg_run_lang_sig.loc[mask, p_col + '_sig'] = reject

avg_run_lang_sig = avg_run_lang_sig.sort_values(
    ['category', 'mean_shift'], ascending=[True, False]
).reset_index(drop=True)

display(avg_run_lang_sig)

# ------------------------------------------------------------
# 8) Matrices aligned to avg_run_lang_matrix
#    Main p-value choice: Wilcoxon FDR-adjusted
# ------------------------------------------------------------
p_col_main = 'wilcoxon_p_fdr'

if not avg_run_lang_matrix.empty:
    pval_matrix = (
        avg_run_lang_sig
        .pivot(index='category', columns='language', values=p_col_main)
        .reindex(index=avg_run_lang_matrix.index, columns=avg_run_lang_matrix.columns)
    )

    n_matrix = (
        avg_run_lang_sig
        .pivot(index='category', columns='language', values='n_titles')
        .reindex(index=avg_run_lang_matrix.index, columns=avg_run_lang_matrix.columns)
    )
else:
    pval_matrix = pd.DataFrame()
    n_matrix = pd.DataFrame()

def p_to_stars(p):
    if pd.isna(p):
        return ''
    elif p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

if not pval_matrix.empty:
    stars_matrix = pval_matrix.applymap(p_to_stars)
else:
    stars_matrix = pd.DataFrame()

# Annotated matrix as strings: "value***"
if not avg_run_lang_matrix.empty:
    avg_run_lang_matrix_annot = avg_run_lang_matrix.copy().astype(object)
    for r in avg_run_lang_matrix.index:
        for c in avg_run_lang_matrix.columns:
            val = avg_run_lang_matrix.loc[r, c]
            star = stars_matrix.loc[r, c] if not stars_matrix.empty else ''
            if pd.isna(val):
                avg_run_lang_matrix_annot.loc[r, c] = np.nan
            else:
                avg_run_lang_matrix_annot.loc[r, c] = f"{val:.2f}{star}"
else:
    avg_run_lang_matrix_annot = pd.DataFrame()

display(pval_matrix.round(4) if not pval_matrix.empty else pval_matrix)
display(stars_matrix)
display(avg_run_lang_matrix_annot)


# ------------------------------------------------------------
# 9) Heatmap with significance stars
#     Annotation uses FDR-adjusted Wilcoxon p-values
# ------------------------------------------------------------
if not avg_run_lang_matrix.empty:
    annot_matrix = avg_run_lang_matrix.copy().astype(object)
    for r in avg_run_lang_matrix.index:
        for c in avg_run_lang_matrix.columns:
            val = avg_run_lang_matrix.loc[r, c]
            star = stars_matrix.loc[r, c] if not stars_matrix.empty else ''
            annot_matrix.loc[r, c] = '' if pd.isna(val) else f'{val:.2f}\n{star}'

    plt.figure(figsize=(11, 7), dpi=1200)
    sns.heatmap(
        avg_run_lang_matrix,
        annot=annot_matrix,
        fmt='',
        cmap='coolwarm',
        center=0
    )
    plt.title('Avg Level Shift vs Others by Language and Category\n(* FDR-adjusted Wilcoxon p < .05)')
    plt.xlabel('Language')
    plt.ylabel('Category')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

## Significance tests for language disparity

We test whether average suitability level differs by language (using per-title, per-category mean across runs). First we run a two-way ANOVA with language plus category main effects. Then we run one-way ANOVA inside each category and FDR-correct the language p-values across categories.

In [ ]:
# Significance tests: language main effect and language x category differences on avg_level
import numpy as np
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests

if avg_run_multi.empty:
    print("avg_run_multi is empty; cannot run significance tests.")
else:
    anova_df = avg_run_multi[['rss_title', 'language', 'category', 'avg_level']].dropna()

    # Canonicalize rare model-output category typos before OLS.
    # Without this, C(category) sees typo variants as extra categories
    # (e.g., 20 levels instead of the intended 15 GARM categories).
    category_corrections = {
        'obsccenity_profanity': 'obscenity_profanity',
        'obscene_profanity': 'obscenity_profanity',
        'obsceny_profanity': 'obscenity_profanity',
        'obscity_profanity': 'obscenity_profanity',
        'sensitive_socil_issues': 'sensitive_social_issues',
    }
    canonical_categories = [
        'adult_sexual', 'alcohol', 'arms_ammunition', 'crime',
        'death_injury_military', 'hate_speech', 'illegal_drugs',
        'misinformation', 'obscenity_profanity', 'online_piracy',
        'other', 'sensitive_social_issues', 'spam_malware',
        'terrorism', 'tobacco_vaping',
    ]
    anova_df = anova_df.copy()
    anova_df['category'] = anova_df['category'].replace(category_corrections)
    unexpected_categories = sorted(set(anova_df['category']) - set(canonical_categories))
    if unexpected_categories:
        print('Dropping unexpected categories:', unexpected_categories)
    anova_df = anova_df[anova_df['category'].isin(canonical_categories)].copy()
    print(f"ANOVA category levels after correction: {anova_df['category'].nunique()}")

    # two-way ANOVA: language effect controlling for category differences
    overall_model = ols('avg_level ~ C(language) + C(category)', data=anova_df).fit()
    overall_anova = anova_lm(overall_model, typ=2)
    display(overall_anova.style.format('{:.4f}').set_caption('Two-way ANOVA: language effect on avg_level (controls category)'))

    # per-category one-way ANOVA across languages (FDR-corrected p-values)
    cat_rows = []
    for cat, g in anova_df.groupby('category'):
        langs = g['language'].nunique()
        if langs < 2 or len(g) < 3:
            continue
        model = ols('avg_level ~ C(language)', data=g).fit()
        table = anova_lm(model, typ=2)
        p = table.loc['C(language)', 'PR(>F)'] if 'C(language)' in table.index else np.nan
        cat_rows.append({
            'category': cat,
            'languages': langs,
            'titles': g['rss_title'].nunique(),
            'rows': len(g),
            'p_language': p,
        })

    cat_anova = pd.DataFrame(cat_rows)
    if cat_anova.empty:
        print('No categories had >=2 languages to test.')
    else:
        cat_anova['p_adj_fdr'] = multipletests(cat_anova['p_language'], method='fdr_bh')[1]
        cat_anova = cat_anova.sort_values('p_language')
        display(cat_anova.style.format({'p_language': '{:.4f}', 'p_adj_fdr': '{:.4f}'}).set_caption('Per-category ANOVA: language effect on avg_level (FDR corrected)'))
